# Sync JobAI with GitHub

Run this notebook anytime you want to update your Google Drive project with the latest commits, pre-trained models, notebooks, and bugfixes from the official GitHub repository (`origin/main`).

It automatically handles:
1. Connecting to Google Drive (`/content/drive/MyDrive/JobAI`).
2. Initializing Git if your upload missed the hidden `.git` folder.
3. Fetching and pulling all new files safely without touching your downloaded 8 GB base model.

In [ ]:
from google.colab import drive
import os, subprocess, sys
from pathlib import Path

# 1. Mount Google Drive and enter JobAI project
drive.mount("/content/drive")
jobai_dir = Path("/content/drive/MyDrive/JobAI")
assert jobai_dir.is_dir(), f"Project directory not found: {jobai_dir}. Please upload JobAI to Google Drive."
os.chdir(jobai_dir)
print(f"Current working directory: {os.getcwd()}")

def run_cmd(cmd):
    print(f"> {cmd}")
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if res.stdout.strip():
        print(res.stdout.strip())
    if res.stderr.strip() and res.returncode != 0:
        print(f"Stderr: {res.stderr.strip()}")
    return res.returncode

# 2. Check / initialize Git repository
if not Path(".git").is_dir():
    print("\n[INFO] No .git directory found. Initializing Git and linking to GitHub...")
    run_cmd("git init")
    run_cmd("git remote add origin https://github.com/Lion504/JobAI.git")
else:
    print("\n[INFO] Existing .git directory found. Verifying remote origin...")
    run_cmd("git remote set-url origin https://github.com/Lion504/JobAI.git")

# 3. Fetch latest updates from main branch
print("\n[INFO] Fetching latest commits from GitHub...")
run_cmd("git fetch origin main")

# 4. Pull changes safely
print("\n[INFO] Pulling updates...")
code = run_cmd("git pull origin main --no-rebase")
if code != 0:
    print("\n[NOTICE] Uncommitted local changes detected. Stashing changes to pull cleanly...")
    run_cmd("git stash")
    run_cmd("git pull origin main --no-rebase")

# 5. Confirmation and status check
print("\n" + "="*50)
print("Repository successfully synchronized!")
run_cmd("git log -1 --oneline")
print("="*50)
print("Checking available model adapters:")
run_cmd("ls -la models/adapters/")